# ChurnShield — 02. Feature Engineering

Build the leakage-safe preprocessing pipeline used by all three models.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))
from src.preprocessing import run_cleaning_pipeline, get_feature_target, NUMERIC_FEATURES, CATEGORICAL_FEATURES, build_preprocessor
from sklearn.model_selection import train_test_split

df = run_cleaning_pipeline(save=True)
X, y = get_feature_target(df)
print("Target distribution:")
print(y.value_counts())

## 1. Feature groups

In [ ]:
print("Numeric:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)

## 2. Train / validation / test split

Use 60% train, 20% validation, 20% held-out test, stratified by churn.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

## 3. ColumnTransformer

In [ ]:
preprocessor = build_preprocessor()
print(preprocessor)

## 4. Leakage check

The preprocessor is intended to be fitted inside the full Scikit-Learn pipeline so that imputation, scaling and one-hot encoding are learned only from the training partition.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear", random_state=42))
])
pipe.fit(X_train, y_train)
print("Pipeline fit complete.")
print("Test probability sample:", pipe.predict_proba(X_test.head(3))[:,1])

## Feature-engineering summary

Numeric variables are median-imputed and standardized. Categorical variables are imputed and one-hot encoded. `customerID` is excluded from model input.